# P10 평가 2/2 — CPU probe·전이·통계

**학습은 다시 실행하지 않습니다. P10은 반환·원본 재현 감사 전까지 미완료입니다.**
입력 ZIP `v1_4_p10_update_evaluation_r1.zip`을 내 드라이브 `boolean_interp_v1_4/`에 올리세요. 기존 학습 출력 `P10_update_l3_r1`을 유지합니다. 평가 출력은 `P10_update_evaluation_r1`입니다.

첫 번째 노트북의 features_complete.json이 있어야 합니다. 런타임 유형을 **하드웨어 가속기 없음**으로 선택하세요. 1,248개 fitting 과제·2,016개 semantic 보고를 순차 저장합니다. 계산량이 많으므로 여러 세션으로 나누어도 됩니다. 고정 예산·support·lambda·bootstrap 수를 줄이지 않습니다.

In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,zipfile,sys,subprocess,os,time,shutil
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
BUNDLE=BASE/'v1_4_p10_update_evaluation_r1.zip'
def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert digest(BUNDLE)=='69bb4191321ed5228cdb5a5dd0e90f9cbeaf1e506aedd1a85a87c9ef87b22e1e', 'ZIP checksum 불일치'
ROOT=Path('/content/MI_P10_evaluation_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for n in z.namelist():assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
SOURCE=BASE/'P10_update_l3_r1'
OUTPUT=BASE/'P10_update_evaluation_r1';OUTPUT.mkdir(exist_ok=True)

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout);print(log.stdout);log.check_returncode()
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2';os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.path.insert(0,str(ROOT))
from interp_v1_4.p10_evaluation import verify,export
verify(ROOT)
def run(args):
    path=OUTPUT/'logs'/f'{time.time_ns()}.txt';path.parent.mkdir(exist_ok=True)
    with path.open('w') as log:
        p=subprocess.Popen([sys.executable,'-u']+args,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in p.stdout:print(line,end='');log.write(line);log.flush()
            rc=p.wait()
        except BaseException:p.terminate();p.wait();raise
    assert rc==0, f'실행 실패: {path}; 마지막 셀에서 증빙을 반환하세요'
run(['-m','pytest','tests_v1_4/test_p10_evaluation.py','-q'])
MAX_SECONDS=7200  # 호출별 chunk/task 경계에서 정지; 중단 후 같은 셀 재실행

In [ ]:
assert (OUTPUT/'features_complete.json').exists(), '1/2 T4 노트북부터 완료하세요'
run(['-m','interp_v1_4.p10_evaluation','select','--root',str(ROOT),'--output',str(OUTPUT),'--max-seconds',str(MAX_SECONDS)])

모든 train/validation 선택을 저장한 뒤에만 test 평가를 시작합니다. 선택 미완료이면 위 셀을 반복하세요. 과제별 source-domain 표준화·ranking·lambda·threshold를 다시 fit하며, Update dictionary 자체는 전체 train으로 학습했습니다. 따라서 전이는 **probe의 감독 전이**입니다.

In [ ]:
run(['-m','interp_v1_4.p10_evaluation','evaluate','--root',str(ROOT),'--output',str(OUTPUT),'--max-seconds',str(MAX_SECONDS)])

## 결과 반환·중단 증빙 보존
완료 또는 실패/중단 시 실행합니다. cache/arrays/latents는 Drive에 유지하고 나머지 환경·로그·fitting trace·예측·의미/fidelity 통계를 ZIP으로 반환합니다. CPU 평가 중 PAUSED이면 해당 셀을 반복하세요. 실제 GPU/CPU 환경·소요 시간과 모든 수치가 반환 감사 대상입니다. 최종적으로 ZIP과 checksum JSON을 보내주세요.

In [ ]:
archive=Path(f'/content/v1_4_p10_evaluation_return_{time.time_ns()}.zip')
export(OUTPUT,archive)
back=OUTPUT/'exports';back.mkdir(exist_ok=True)
shutil.copyfile(archive,back/archive.name);shutil.copyfile(archive.with_suffix('.sha256.json'),back/archive.with_suffix('.sha256.json').name)
print('checksum',digest(archive));files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))